# 02b - Probability & Inference Analyst: Inferensi Statistik  (Minggu 3, alpha = 0.05)

**Peran:** Probability & Inference Analyst
**Dikerjakan oleh:** _(isi nama)_

### Cerita singkat tugasmu
Data kita hanya **sampel** (345 pengukuran). Kita ingin menyimpulkan hal tentang **populasi** (kondisi udara kota sesungguhnya).
Inferensi = "menyimpulkan yang lebih luas dari data yang terbatas, sambil jujur soal ketidakpastiannya".

| Bagian | Pertanyaan | Alat |
|---|---|---|
| A | Berapa kisaran wajar rata-rata PM2.5 & proporsi udara Tidak Sehat di populasi? | Confidence Interval 95% |
| B | PM2.5 Hari Kerja vs Akhir Pekan beda nyata? | Uji-t dua sampel independen |
| C | PM2.5 di 3 zona lokasi beda nyata? | One-Way ANOVA + Tukey |
| D | Status udara berkaitan dengan Jam Sibuk? | Chi-Square independensi |

**Untuk SETIAP uji wajib tertulis:** H0, H1, statistik uji, p-value, keputusan, interpretasi bahasa sehari-hari.

### Resep dasar uji hipotesis (hafalkan ini)
1. **H0** (hipotesis nol) = "tidak ada perbedaan / tidak ada hubungan". Posisi awal yang kita anggap benar.
2. **H1** = "ada perbedaan / ada hubungan". Yang ingin kita buktikan.
3. Hitung **statistik uji** (t, F, atau chi-kuadrat) dan **p-value**.
4. **p-value** = peluang mendapat hasil sekuat ini (atau lebih ekstrem) **jika H0 benar**.
5. **Keputusan:** p < 0.05 -> **tolak H0** (ada bukti kuat). p >= 0.05 -> **gagal tolak H0** (bukti belum cukup).
   ⚠️ Jangan bilang "H0 diterima/terbukti benar" - kita hanya *gagal menolak*.

In [ ]:
# ============ SETUP (jalankan sel ini PERTAMA) ============
# Satu-satunya yang boleh diubah: GITHUB_USER dan GITHUB_REPO (diisi sekali oleh Project Lead).
import os, sys, subprocess
GITHUB_USER = "USERNAME"      # <- username GitHub pemilik repo
GITHUB_REPO = "NAMA-REPO"     # <- nama repository
BRANCH = "main"

if "google.colab" in sys.modules:                      # kalau dijalankan di Google Colab
    PROYEK = "/content/proyek"
    if not os.path.exists(PROYEK):
        token = ""
        try:                                           # opsional: untuk repo PRIVATE (lihat docs/02_...)
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN") + "@"
        except Exception:
            pass
        url = f"https://{token}github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, PROYEK],
                           capture_output=True, text=True)
        if r.returncode != 0:
            pesan = (r.stderr or "").replace(token.rstrip("@"), "***") if token else (r.stderr or "")
            raise RuntimeError("Gagal clone repo. Cek GITHUB_USER/GITHUB_REPO, atau lihat bagian "
                               "'Troubleshooting' di docs/02_panduan_github_colab.md\n" + pesan)
    else:
        subprocess.run(["git", "-C", PROYEK, "pull"], capture_output=True)   # ambil update terbaru
    os.chdir(PROYEK)
elif os.path.basename(os.getcwd()) == "notebooks":    # kalau dijalankan di Jupyter lokal
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from src.helpers import *
sns.set_theme(style="whitegrid", font_scale=1.0)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Setup selesai. Folder kerja:", os.getcwd())

In [ ]:
df = muat_data_bersih()
N = len(df)
VAR = "Indeks_PM25"
hasil_uji = {}      # tempat merangkum hasil semua uji
df.head()

## A. Selang Kepercayaan / Confidence Interval 95% (Minggu 3a)

CI = rentang yang "kemungkinan besar" memuat nilai sebenarnya di populasi.
**Cara membaca benar:** *"Kami 95% yakin rata-rata PM2.5 populasi berada di antara A dan B."*
(Secara teknis: jika pengambilan sampel diulang berkali-kali, 95% dari CI yang dibuat akan memuat nilai populasi.)

### A1. CI rata-rata PM2.5
$$\bar{x} \pm t_{\alpha/2,\,n-1} \cdot \frac{s}{\sqrt{n}}$$

In [ ]:
x = df[VAR]
n = len(x)
xbar, s = x.mean(), x.std(ddof=1)
se = s / np.sqrt(n)                              # standard error
t_krit = stats.t.ppf(1 - ALPHA / 2, df=n - 1)    # nilai t kritis
bb, ba = xbar - t_krit * se, xbar + t_krit * se
print(f"n = {n} | mean = {xbar:.3f} | s = {s:.3f} | SE = {se:.3f} | t kritis = {t_krit:.3f}")
print(f"CI 95% rata-rata PM2.5 = [{bb:.2f} ; {ba:.2f}]")

# Cek dengan fungsi scipy (harus sama)
cek = stats.t.interval(0.95, df=n - 1, loc=xbar, scale=se)
print("Cek scipy:", np.round(cek, 2))
hasil_uji["CI_mean"] = (xbar, bb, ba)

### A2. CI proporsi kategori "Tidak Sehat"
$$\hat{p} \pm z_{\alpha/2}\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$
(pendekatan normal / Wald; syarat: $n\hat{p} \ge 5$ dan $n(1-\hat{p}) \ge 5$). Kita bandingkan dengan metode Wilson yang lebih akurat untuk proporsi kecil.

In [ ]:
from statsmodels.stats.proportion import proportion_confint

kasus = int((df["Status_Udara"] == "Tidak Sehat").sum())
p_hat = kasus / N
z_krit = stats.norm.ppf(1 - ALPHA / 2)
se_p = np.sqrt(p_hat * (1 - p_hat) / N)
print(f"p topi = {kasus}/{N} = {p_hat:.4f}")
print(f"Syarat: n*p = {N*p_hat:.1f} (>=5?)  n*(1-p) = {N*(1-p_hat):.1f} (>=5?)")
wald = (p_hat - z_krit * se_p, p_hat + z_krit * se_p)
wilson = proportion_confint(kasus, N, alpha=ALPHA, method="wilson")
print(f"CI 95% Wald   : [{wald[0]:.4f} ; {wald[1]:.4f}]  -> [{wald[0]*100:.2f}% ; {wald[1]*100:.2f}%]")
print(f"CI 95% Wilson : [{wilson[0]:.4f} ; {wilson[1]:.4f}]")

tabel_ci = pd.DataFrame([
    ["Rata-rata PM2.5", round(xbar, 3), round(bb, 3), round(ba, 3), "t-interval"],
    ["Proporsi Tidak Sehat", round(p_hat, 4), round(wald[0], 4), round(wald[1], 4), "Wald (normal)"],
    ["Proporsi Tidak Sehat", round(p_hat, 4), round(wilson[0], 4), round(wilson[1], 4), "Wilson (pembanding)"],
], columns=["Parameter", "Estimasi", "Batas_Bawah", "Batas_Atas", "Metode"])
simpan_tabel(tabel_ci, "03_confidence_interval", index=False)
tabel_ci

✏️ **ISI - Interpretasi CI (bahasa sehari-hari):**
_"Rata-rata PM2.5 di kota ini diperkirakan berada antara ... dan ... (95% yakin). Sebanyak sekitar ...% hingga ...% pengukuran berstatus Tidak Sehat."_
Tambahkan: apakah batas atas rata-rata itu masih di bawah 65 (ambang Tidak Sehat)? Apa artinya?

_(tulis di sini)_

## B. Uji-t Dua Sampel Independen (Minggu 3b-1)

**Pertanyaan:** apakah rata-rata `Indeks_PM25` berbeda antara **Hari Kerja** dan **Akhir Pekan**?

- **H0:** $\mu_{Hari\ Kerja} = \mu_{Akhir\ Pekan}$ (rata-rata PM2.5 sama)
- **H1:** $\mu_{Hari\ Kerja} \ne \mu_{Akhir\ Pekan}$ (rata-rata berbeda) -> uji **dua arah**
- $\alpha = 0.05$

In [ ]:
g1 = df.loc[df["Jenis_Hari"] == "Hari Kerja", VAR]
g2 = df.loc[df["Jenis_Hari"] == "Akhir Pekan", VAR]
desk = pd.DataFrame({"n": [len(g1), len(g2)], "Mean": [g1.mean(), g2.mean()],
                     "Std": [g1.std(), g2.std()]}, index=["Hari Kerja", "Akhir Pekan"]).round(2)
desk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df, x="Jenis_Hari", y=VAR, hue="Jenis_Hari", order=["Hari Kerja", "Akhir Pekan"],
            palette={"Hari Kerja": "#e07b39", "Akhir Pekan": "#4a90c2"}, legend=False, ax=axes[0])
axes[0].set_title("PM2.5 menurut jenis hari"); axes[0].set_xlabel("")
sns.histplot(data=df, x=VAR, hue="Jenis_Hari", bins=25, stat="density", common_norm=False, alpha=0.5, ax=axes[1])
axes[1].set_title("Sebaran PM2.5 per kelompok")
plt.tight_layout()
simpan_gambar(fig, "03_ttest_boxplot")
plt.show()

### Cek asumsi sebelum uji-t
1. **Independen** - kelompok terdiri dari observasi berbeda. ✔
2. **Normal** - dengan n besar (>30 per kelompok) Teorema Limit Pusat membuat uji-t cukup kuat. Tetap cek visual (histogram di atas).
3. **Varians sama?** -> **Uji Levene** (H0: varians kedua kelompok sama).
   - Levene p >= 0.05 -> pakai **Student t-test** (varians sama / pooled).
   - Levene p < 0.05 -> pakai **Welch t-test** (varians beda).

In [ ]:
lev_stat, lev_p = stats.levene(g1, g2)
sama = lev_p >= ALPHA
print(f"Levene: stat = {lev_stat:.3f}, p = {lev_p:.4f} -> varians {'SAMA' if sama else 'BERBEDA'}")

t_stat, p_t = stats.ttest_ind(g1, g2, equal_var=sama)        # equal_var=True: Student, False: Welch
if sama:
    df_t = len(g1) + len(g2) - 2
else:       # derajat bebas Welch-Satterthwaite
    v1, v2 = g1.var(), g2.var(); n1, n2 = len(g1), len(g2)
    df_t = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))

selisih = g1.mean() - g2.mean()
sp = np.sqrt(((len(g1)-1)*g1.var() + (len(g2)-1)*g2.var()) / (len(g1)+len(g2)-2))
cohen_d = selisih / sp
print(f"Metode          : {'Student (varians sama)' if sama else 'Welch (varians beda)'}")
print(f"t = {t_stat:.3f}, df = {df_t:.1f}, p-value = {p_t:.3g}")
print(f"Selisih rata-rata (Kerja - Akhir Pekan) = {selisih:.2f}")
print(f"Cohen's d = {cohen_d:.2f}  (0.2 kecil, 0.5 sedang, 0.8 besar)")
print("KEPUTUSAN:", "TOLAK H0 (p < 0.05)" if p_t < ALPHA else "GAGAL TOLAK H0 (p >= 0.05)")

tabel_t = pd.DataFrame([{
    "Uji": "t independen " + ("(Student)" if sama else "(Welch)"),
    "Mean_Hari_Kerja": round(g1.mean(), 2), "Mean_Akhir_Pekan": round(g2.mean(), 2),
    "Selisih": round(selisih, 2), "t": round(t_stat, 3), "df": round(df_t, 1),
    "p_value": p_t, "Cohen_d": round(cohen_d, 2), "Levene_p": round(lev_p, 4),
    "Keputusan": "Tolak H0" if p_t < ALPHA else "Gagal tolak H0"}])
simpan_tabel(tabel_t, "03_ttest_hasil", index=False)

In [ ]:
# Cek ketahanan: uji non-parametrik (tidak butuh asumsi normal). Hasilnya seharusnya searah.
u, p_u = stats.mannwhitneyu(g1, g2, alternative="two-sided")
print(f"Mann-Whitney U: U = {u:.0f}, p = {p_u:.3g}  (pembanding saja; yang dilaporkan utama tetap uji-t)")

✏️ **ISI - Tulis lengkap (akan dipakai Technical Writer):**

| Komponen | Isi |
|---|---|
| H0 | _(tulis)_ |
| H1 | _(tulis)_ |
| Statistik uji | t = ..., df = ... |
| p-value | ... |
| Keputusan | Tolak / Gagal tolak H0 pada alpha 0.05 |
| Interpretasi sehari-hari | _"Rata-rata PM2.5 pada hari kerja (...) (lebih tinggi / tidak berbeda nyata) dibanding akhir pekan (...). Selisihnya sekitar ... poin. Artinya ..."_ |

## C. One-Way ANOVA (Minggu 3b-2)

**Pertanyaan:** apakah rata-rata `Indeks_PM25` berbeda di ketiga **Zona Lokasi** (Pusat Kota, Kawasan Kampus, Pinggiran)?

Kenapa tidak memakai 3 kali uji-t? Setiap uji punya risiko salah 5%; kalau diulang 3 kali, risiko salah total membengkak. ANOVA menguji **semua kelompok sekaligus**.

- **H0:** $\mu_{Pusat} = \mu_{Kampus} = \mu_{Pinggiran}$ (ketiganya sama)
- **H1:** **minimal satu** zona rata-ratanya berbeda
- Statistik uji **F** = (variasi *antar* kelompok) / (variasi *di dalam* kelompok). F besar -> kelompok memang berbeda.

In [ ]:
kel = [df.loc[df["Zona_Lokasi"] == z, VAR] for z in URUTAN_ZONA]
desk_z = df.groupby("Zona_Lokasi")[VAR].agg(["count", "mean", "std"]).loc[URUTAN_ZONA].round(2)
display(desk_z)

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(data=df, x="Zona_Lokasi", y=VAR, hue="Zona_Lokasi", order=URUTAN_ZONA, palette=WARNA_ZONA, legend=False, ax=ax)
ax.set_title("PM2.5 menurut zona lokasi"); ax.set_xlabel("")
simpan_gambar(fig, "03_anova_boxplot")
plt.show()

In [ ]:
# Asumsi 1: varians sama (Levene)
lev_s, lev_pz = stats.levene(*kel)
print(f"Levene antar zona: stat = {lev_s:.3f}, p = {lev_pz:.4f} -> varians {'SAMA (ANOVA aman)' if lev_pz >= ALPHA else 'BERBEDA (pertimbangkan Welch ANOVA / Kruskal-Wallis)'}")

# ANOVA
F, p_f = stats.f_oneway(*kel)
k_grup, n_tot = len(kel), len(df)
df_antar, df_dalam = k_grup - 1, n_tot - k_grup
# Ukuran efek: eta kuadrat = SS_antar / SS_total
grand = df[VAR].mean()
ss_antar = sum(len(g) * (g.mean() - grand) ** 2 for g in kel)
ss_total = ((df[VAR] - grand) ** 2).sum()
eta2 = ss_antar / ss_total
print(f"F({df_antar}, {df_dalam}) = {F:.3f}, p-value = {p_f:.3g}")
print(f"Eta kuadrat = {eta2:.3f}  -> zona menjelaskan ~{eta2*100:.1f}% variasi PM2.5  (0.01 kecil, 0.06 sedang, 0.14 besar)")
print("KEPUTUSAN:", "TOLAK H0 -> ada perbedaan -> lanjut Tukey" if p_f < ALPHA else "GAGAL TOLAK H0 -> tidak perlu uji lanjut")

tabel_anova = pd.DataFrame([{"Sumber": "Antar zona", "df": df_antar, "SS": round(ss_antar, 2), "MS": round(ss_antar/df_antar, 2), "F": round(F, 3), "p_value": p_f},
                            {"Sumber": "Dalam zona", "df": df_dalam, "SS": round(ss_total-ss_antar, 2), "MS": round((ss_total-ss_antar)/df_dalam, 2), "F": np.nan, "p_value": np.nan},
                            {"Sumber": "Total", "df": n_tot-1, "SS": round(ss_total, 2), "MS": np.nan, "F": np.nan, "p_value": np.nan}])
simpan_tabel(tabel_anova, "03_anova_hasil", index=False)
tabel_anova

### Uji lanjut Post-Hoc Tukey HSD
ANOVA hanya bilang "ada yang beda". **Tukey** memberi tahu **pasangan zona mana** yang beda, dengan mengendalikan risiko salah total di 5%.
Kolom `reject = True` berarti pasangan itu berbeda nyata. (Hanya relevan jika ANOVA signifikan.)

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
tukey = pairwise_tukeyhsd(endog=df[VAR], groups=df["Zona_Lokasi"], alpha=ALPHA)
print(tukey)
tabel_tukey = pd.DataFrame(tukey._results_table.data[1:], columns=tukey._results_table.data[0])
simpan_tabel(tabel_tukey, "03_tukey_hasil", index=False)

In [ ]:
# Pembanding ketahanan (tidak butuh normal & varians sama)
H, p_kw = stats.kruskal(*kel)
print(f"Kruskal-Wallis: H = {H:.2f}, p = {p_kw:.3g}  (pembanding saja)")

✏️ **ISI:**

| Komponen | Isi |
|---|---|
| H0 | _(tulis)_ |
| H1 | _(tulis)_ |
| Statistik uji | F(..., ...) = ... |
| p-value | ... |
| Keputusan | Tolak / Gagal tolak H0 |
| Tukey: pasangan yang berbeda | _(mis. "Pusat Kota vs Pinggiran: beda ... poin, p = ...")_ |
| Interpretasi sehari-hari | _(zona mana paling kotor, mana paling bersih, seberapa besar bedanya)_ |

## D. Chi-Square Test of Independence (Minggu 3b-3)

**Pertanyaan:** apakah `Status_Udara` berkaitan dengan `Jam_Sibuk`? (dua variabel **kategori**)

- **H0:** Status udara **independen** (tidak berkaitan) dengan jam sibuk
- **H1:** Status udara **berkaitan** dengan jam sibuk
- Ide: bandingkan **frekuensi observasi** (data nyata) dengan **frekuensi ekspektasi** (yang akan muncul jika H0 benar). Makin jauh bedanya, makin besar $\chi^2$.

$$E_{ij} = \frac{(\text{total baris } i)(\text{total kolom } j)}{N}, \qquad \chi^2 = \sum \frac{(O - E)^2}{E}$$

In [ ]:
observasi = pd.crosstab(df["Status_Udara"], df["Jam_Sibuk"]).reindex(URUTAN_STATUS)
chi2, p_chi, dof, ekspektasi = stats.chi2_contingency(observasi)
ekspektasi = pd.DataFrame(ekspektasi, index=observasi.index, columns=observasi.columns)

print("=== Frekuensi OBSERVASI ===");  display(observasi)
print("=== Frekuensi EKSPEKTASI (jika H0 benar) ==="); display(ekspektasi.round(2))
print("Syarat: semua ekspektasi >= 5 ?", bool((ekspektasi >= 5).all().all()), "| terkecil =", round(ekspektasi.values.min(), 2))

In [ ]:
n_chi = observasi.values.sum()
cramers_v = np.sqrt(chi2 / (n_chi * (min(observasi.shape) - 1)))
print(f"Chi-square = {chi2:.3f}, df = {dof}, p-value = {p_chi:.4f}")
print(f"Cramer's V = {cramers_v:.3f}  (0.1 lemah, 0.3 sedang, 0.5 kuat)")
print("KEPUTUSAN:", "TOLAK H0 -> ada hubungan" if p_chi < ALPHA else "GAGAL TOLAK H0 -> tidak ada bukti hubungan")

simpan_tabel(observasi, "03_chi2_observasi")
simpan_tabel(ekspektasi.round(2), "03_chi2_ekspektasi")
tabel_chi = pd.DataFrame([{"Uji": "Chi-Square independensi (Status x Jam Sibuk)", "chi2": round(chi2, 3), "df": dof,
                           "p_value": p_chi, "Cramers_V": round(cramers_v, 3),
                           "Keputusan": "Tolak H0" if p_chi < ALPHA else "Gagal tolak H0"}])
simpan_tabel(tabel_chi, "03_chi2_hasil", index=False)

# Persentase per kolom untuk interpretasi
persen = (observasi / observasi.sum() * 100).round(1)
persen

✏️ **ISI:**

| Komponen | Isi |
|---|---|
| H0 | _(tulis)_ |
| H1 | _(tulis)_ |
| Statistik uji | chi2 = ..., df = ... |
| p-value | ... |
| Keputusan | Tolak / Gagal tolak H0 |
| Interpretasi sehari-hari | _(Tabel persen di atas menunjukkan ...; apakah jam sibuk benar-benar membedakan komposisi status udara?)_ |

💡 **Jika hasilnya TIDAK signifikan, itu bukan kegagalan!** Itu temuan: *"jam sibuk saja tidak cukup menjelaskan status udara; faktor lain (mis. zona lokasi) lebih berperan."* Coba (opsional) uji juga `Status_Udara x Zona_Lokasi` dengan kode yang sama untuk melihat bedanya.

In [ ]:
# OPSIONAL: bandingkan dengan Status_Udara x Zona_Lokasi
obs2 = pd.crosstab(df["Zona_Lokasi"], df["Status_Udara"])[URUTAN_STATUS].reindex(URUTAN_ZONA)
chi2_b, p_b, dof_b, exp_b = stats.chi2_contingency(obs2)
print(f"Status x Zona: chi2 = {chi2_b:.2f}, df = {dof_b}, p = {p_b:.3g}")
print("Catatan: ada sel ekspektasi < 5?", bool((exp_b < 5).any()), "(kalau ya, hati-hati menafsirkan)")
obs2

## Ringkasan seluruh uji (untuk Technical Writer & Project Lead)

In [ ]:
ringkas = pd.DataFrame([
    ["t independen (Hari Kerja vs Akhir Pekan)", f"t = {t_stat:.2f}", p_t, "Tolak H0" if p_t < ALPHA else "Gagal tolak H0"],
    ["One-Way ANOVA (3 zona)", f"F = {F:.2f}", p_f, "Tolak H0" if p_f < ALPHA else "Gagal tolak H0"],
    ["Chi-Square (Status x Jam Sibuk)", f"chi2 = {chi2:.2f}", p_chi, "Tolak H0" if p_chi < ALPHA else "Gagal tolak H0"],
], columns=["Uji", "Statistik", "p_value", "Keputusan (alpha 0.05)"])
ringkas["p_value"] = ringkas["p_value"].map(lambda v: f"{v:.3g}")
simpan_tabel(ringkas, "03_ringkasan_uji_hipotesis", index=False)
ringkas

## Serahkan hasil

In [ ]:
unduh_hasil("hasil_02b_inference")

### ✅ Checklist selesai (Minggu 3)
- [ ] CI 95% rata-rata PM2.5 **dan** CI proporsi Tidak Sehat + interpretasi
- [ ] t-test: H0, H1, t, p, keputusan, interpretasi (tabel isian terisi)
- [ ] ANOVA: H0, H1, F, p, keputusan; **Tukey** (jika signifikan) + interpretasi
- [ ] Chi-square: tabel observasi **dan** ekspektasi, chi2, p, keputusan, interpretasi
- [ ] Semua file sudah di-upload ke `outputs/`
- [ ] Aku bisa menjelaskan tanpa membaca: apa itu p-value, kenapa pakai ANOVA bukan 3x uji-t, apa arti "gagal tolak H0"